# **Home Exercise 1 on Text Classification**
Implement a Recurrent Neural Network model (Vanilla RNN, GRU, and LSTM) to predict whether a review is positive or negative.

**Data**: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) (the last 10% of rows serve as the test set).
Compare the performance of the three models.


In [1]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
import os
import sys
import numpy as np
import pandas as pd

import re
from collections import Counter
import matplotlib.pyplot as plt
from datetime import datetime

print("The last time this project was run is:", datetime.now().strftime("%H:%M:%S %d/%m/%Y"))


if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Using device: CUDA - {gpu_name}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    device = torch.device("cpu")
    print("Using device: CPU (CUDA is not available)")


The last time this project was run is: 03:23:12 20/11/2025
Using device: CUDA - Tesla T4
CUDA Capability: (7, 5)


## Loading data with DataLoader into DataFrame

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab=None, max_len=MAX_SEQ_LEN, is_train=True):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
        
        if is_train:
            self.vocab = self.build_vocab(texts)
        else:
            self.vocab = vocab
            
    def build_vocab(self, texts):
        all_words = []
        for text in texts:
            all_words.extend(tokenizer(clean_text(text)))
        
        # Lấy top từ phổ biến nhất
        count = Counter(all_words)
        sorted_words = count.most_common(MAX_VOCAB_SIZE)
        
        # Tạo dictionary: word -> index (bắt đầu từ 2, chừa 0 cho padding, 1 cho unknown)
        vocab = {w: i+2 for i, (w, c) in enumerate(sorted_words)}
        vocab['<PAD>'] = 0
        vocab['<UNK>'] = 1
        return vocab
    
    def text_to_sequence(self, text):
        tokens = tokenizer(clean_text(text))
        # Map từ sang index, nếu không có trong vocab thì dùng <UNK>
        seq = [self.vocab.get(token, 1) for token in tokens]
        
        # Padding hoặc Truncating
        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq)) # Padding
        else:
            seq = seq[:self.max_len] # Truncating
        return torch.tensor(seq, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Chuyển label 'positive' -> 1, 'negative' -> 0 nếu cần
        # Giả sử dataset gốc là dạng string, ta cần map trước khi đưa vào đây
        # Trong code main bên dưới ta sẽ xử lý việc này.
        
        return self.text_to_sequence(text), torch.tensor(label, dtype=torch.float)